# 02 · 구조화 로그와 요청 연결

목표: 로그를 JSON 필드로 조회하고 요청·추적 ID와 개인정보의 경계를 이해한다. 약12분.

**이 파일의 핵심 질문:** 한 요청의 실패를 원문 없이 어떤 필드 조합으로 찾을 수 있는가?

**구조화 로그**는 JSON 필드로 조회하는 이벤트 기록이다. **request_id**는 요청 식별자, **trace_id**는 관련 작업 묶음의 식별자다. **라벨**은 지표를 나눠 집계하는 분류 값이다.

**수업 중 먼저 실행할 단계**

- 1 비교 요청
- 2 허용 필드
- 3 오류 조회
- 4 사용자 문의와 연결

파일 안의 변수 의존성이 있으므로 처음 실행할 때는 위에서부터 순서대로 진행한다. 핵심 단계에서는 출력을 읽고, 나머지 단계는 빠르게 실행한 뒤 복습 때 살펴본다.

## 시작 전 · 어디에서 무엇을 실행하나요?

1. VS Code에서 `llmops-16week` **폴더 전체**를 연다. 압축을 풀었다면 Notebook 파일만 따로 옮기지 않는다.
2. VS Code의 Python·Jupyter 확장을 설치하고 우측 위 **Select Kernel → Python Environments → .venv / Python 3.11**을 선택한다. 터미널의 가상환경과 Notebook 커널은 별도 선택이다.
3. 아래 **환경 확인 코드 셀부터** `Shift+Enter`로 한 셀씩 실행한다. 이 파일 안에서는 앞에서 만든 변수를 뒤에서 사용한다.
4. 각 단계의 **예상 결과**와 실제 출력을 대조한다. `assert`는 확인 조건이다. 출력이나 오류가 없어도 셀이 완료되면 조건을 통과한 것이다.
5. 셀을 건너뛰어 `NameError`가 나거나 변수를 여러 번 바꿨다면 **Restart Kernel → Run All**로 처음부터 실행한다. 다른 Notebook의 변수는 가져오지 않는다.

처음 환경을 만드는 경우에만 **VS Code 터미널**에서 실행한다. 아래 명령을 Python 코드 셀에 붙여넣지 않는다.

```bash
# macOS / Linux · 저장소 루트
python3.11 -m venv .venv
source .venv/bin/activate
python -m pip install -r requirements.txt
```

Windows PowerShell은 `py -3.11 -m venv .venv`, `.venv\Scripts\Activate.ps1` 순서다. 활성화가 제한된 환경에서는 `.venv\Scripts\python.exe -m pip install -r requirements.txt`처럼 가상환경 Python을 직접 실행한다. 이미 준비된 `.venv`는 그대로 쓴다.

**공통 실습의 실행 방식:** `lab_client`가 메모리 안에 FastAPI 앱을 만들고 시작·종료를 관리한다. HTTPX의 ASGITransport가 같은 프로세스에서 요청을 전달하므로 `uvicorn`, Docker, Ollama, API 키가 필요하지 않다. ASGI는 *Asynchronous Server Gateway Interface*, Python 비동기 웹 앱과 서버 사이의 인터페이스다.

Notebook 앱과 브라우저의 `127.0.0.1:8010` 앱은 **서로 다른 실행**이다. Notebook에서 만든 요청 수가 브라우저·Prometheus에 합쳐지지 않는다. 실제 모델 대신 고정 문장과 대기를 쓰므로 외부 비용은 발생하지 않는다. 실행 시간·요청 ID는 매번 달라져도 정상이다.

In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "week10/lab/app.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import asyncio, json
from dataclasses import replace
from week10.lab.config import Settings
from week10.lab.experiments import lab_client, fixture_check, window_delta, compare_releases
assert sys.version_info[:2] == (3, 11), "Select the Python 3.11 .venv kernel in VS Code"
print("Python:", sys.version.split()[0])
print("실행 환경:", sys.executable)
print("저장소:", ROOT)
print("환경 확인 완료 · 이 Notebook은 별도 서버 없이 실행합니다.")

## 1. 비교 요청

정상·검색 오류·잘못된 입력을 한 번씩 보낸다.

**예상 결과:** 정상200·검색 오류502·빈 입력422. 조건을 통과하면 출력 없이 완료된다.

**결과 해석:** 서버 장애와 입력 검증 실패를 구별할 비교 표본을 만든다. 외부 서버에 요청하는 셀이 아니다.

In [ ]:
async with lab_client() as (app, client):
    ok = await client.post("/api/v1/assist", json={"message":"SYNTHETIC-ONLY"})
    bad = await client.post("/api/v1/assist", json={"message":"예시", "scenario":"retrieval_error"})
    invalid = await client.post("/api/v1/assist", json={"message":" "})
    snapshot = app.state.telemetry.snapshot()
assert [ok.status_code, bad.status_code, invalid.status_code] == [200, 502, 422]

## 2. 허용 필드

원문 대신 route·상태·시간·버전을 남긴다.

**예상 결과:** 로그3건의 JSON. 입력 원문 SYNTHETIC-ONLY는 snapshot에 없다.

**결과 해석:** 상태·기간·버전·연결 ID로 원인을 좁힐 수 있다. 실습의 허용 필드 목록을 읽는다.

In [ ]:
logs = snapshot["logs"]
print(json.dumps(logs, ensure_ascii=False, indent=2))
assert "SYNTHETIC-ONLY" not in json.dumps(snapshot)

## 3. 오류 조회

문자열 검색보다 필드 조건으로 필요한 이벤트를 좁힌다.

**예상 결과:** status>=500인 검색 오류 로그1건.

**결과 해석:** 422는 사용자 입력 검증 실패라 이 필터에 포함되지 않는다. 어떤 실패를 집계하는지 먼저 정의한다.

In [ ]:
errors = [r for r in logs if r["status"] >= 500]
assert len(errors) == 1
errors

## 4. 사용자 문의와 연결

X-Request-ID로 해당 실행을 찾는다. 질문 자체를 로그에 복제할 필요가 없다.

**예상 결과:** 502 응답의 X-Request-ID에 대응하는 로그1건.

**결과 해석:** 문의의 요청 ID와 로그를 연결한다. ID는 실행을 찾는 단서이며 사용자 인증 수단은 아니다.

In [ ]:
rid = bad.headers["X-Request-ID"]
matched = [r for r in logs if r["request_id"] == rid]
assert len(matched) == 1
matched[0]

## 5. Trace 연결

같은 trace_id의 span을 모은다. ID는 인증·권한 증명이 아니다.

**예상 결과:** 같은 trace_id의 retrieve·assist가 ERROR. generate·format은 없다.

**결과 해석:** 검색이 실패했으므로 이후 단계를 실행하지 않았다. 전체 실패와 최초 실패 단계를 함께 읽는다.

In [ ]:
tid = matched[0]["trace_id"]
spans = [s for s in snapshot["spans"] if s["trace_id"] == tid]
[(s["name"], s["status"]) for s in spans]

## 6. 라벨 조합의 증가

예시 계산이다. 요청 ID를 지표 라벨에 넣을 때 1만 건만으로 조합 수가 급증한다.

**예상 결과:** 유한 상태·경로 상한24, 요청 ID를 곱한 가상 상한240000.

**결과 해석:** 서로 다른 라벨 조합마다 시계열이 늘어날 수 있다. 계산 예시이며 실제 저장량·비용 측정은 아니다.

In [ ]:
status_values, route_values, request_ids = 6, 4, 10_000
print("유한 상태·경로 상한:", status_values * route_values)
print("요청 ID까지 곱한 가상 상한:", status_values * route_values * request_ids)
assert "request_id" not in app.state.telemetry.prometheus()

## 7. 메모리 보존 범위

200건 보관과 누적 완료 건수는 다른 값이다. 재시작하면 둘 다 초기화된다.

**예상 결과:** retention의 로그 한도200·Span 한도800과 허용 로그 필드 검사 통과.

**결과 해석:** 최근 로그 보관 건수와 프로세스 수명 동안의 누적 counter는 서로 다른 범위다.

In [ ]:
print(snapshot["retention"])
assert set(logs[0]) == {"event","timestamp","request_id","trace_id","route","status","duration_ms","release","config_id"}

## 직접 바꿔볼 실험 · 선택 복습

1. 오류 필터를 status>=400으로 바꾸고422가 추가되는지 비교한다.
2. 502 대신 정상 응답의 request_id로 조회해 로그1건을 찾는다.
3. 로그에 남길 필드와 지표 라벨로 쓸 필드를 나눠 선택 이유를 적는다.

한 번에 한 조건만 바꾸고 원래 코드와 비교한다. 기대값이 고정된 assert가 있다면 새 실험 조건에 맞는 값도 함께 설명한다.

## 프로젝트 적용 기록

**정리 질문:** 한 요청의 실패를 원문 없이 어떤 필드 조합으로 찾을 수 있는가?

아래 Markdown 셀을 더블클릭해 자신의 결과를 적는다. 다른 파일 제출이나 화면 실행을 자동으로 수행하지 않는다.

| 항목 | 직접 기록 |
|---|---|
| 예상 | 실행 전 예상한 결과 |
| 관찰 | 상태 코드·건수·실패 단계 등 실제 결과 |
| 근거 | 실행한 단계 번호·변경한 조건·캡처 위치 |
| 판단 | 유지·수정·채택·보류 이유 |
| 미검증·다음 변경 | 아직 실행하지 않은 조건과 다음 비교 |

학생 정보나 원문 입력은 공개 저장소에 올리지 않는다. 실제 실행하지 않은 조건은 미측정으로 표시한다.

## 막혔을 때의 확인 순서

| 증상 | 확인·해결 |
|---|---|
| 커널 선택 창만 표시 | Python·Jupyter 확장 설치 후 `.venv`의 Python 선택 |
| `ModuleNotFoundError` | 커널 경로 확인 → 해당 `.venv`로 `python -m pip install -r requirements.txt` |
| 환경 확인에서 저장소를 찾지 못함 | 저장소 전체를 열고 원래 `week10/lab/notebooks` 위치에서 실행 |
| `NameError` | 앞 셀 실행 여부 확인 → Restart Kernel → Run All |
| `asyncio.run() cannot be called...` | Jupyter에서는 제공된 최상위 `await`·`async with`를 그대로 사용 |
| 셀에 출력이 없음 | `assert`만 있는 셀은 오류 없이 끝나면 통과. 왼쪽 실행 번호와 완료 표시 확인 |
| 예상한 422·502·503·504 | 입력 오류·검색 오류·준비 실패·시간 초과를 의도적으로 만든 단계인지 확인 |
| 예상과 다른 `AssertionError` | 바로 위 코드·출력 확인. 변경한 값을 복원하고 처음부터 다시 실행. assert를 지워 통과시키지 않음 |
| 실행 시간·ID가 예시와 다름 | 고정값 비교 대신 상태·부모/자식 관계·분모·검증 결과를 비교 |

실습에서 바꾼 내용은 자신의 사본에 기록한다. 공개 저장소에는 학생 정보·원문 질문·실제 키를 올리지 않는다.